# IoT-NetGuard — Data Preprocessing
**Project:** Anomaly Detection for IoT Cybersecurity  
**Dataset:** IoT-23 (Stratosphere Laboratory)  
**Description:** Loads all 23 capture files, cleans and encodes features, and exports `iot23_combined.csv`.

## 1. Imports

In [ ]:
import os
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

print('pandas version:', pd.__version__)

## 2. Configuration — Dataset Paths
> Update `BASE_PATH` to point to your local IoT-23 dataset folder.

In [ ]:
# ── UPDATE THIS PATH ──────────────────────────────────────────────────────────
BASE_PATH = "D:/5800 Datasets/iot_23_datasets_small/opt/Malware-Project/BigDataset/IoTScenarios"
# ─────────────────────────────────────────────────────────────────────────────

SKIP_ROWS  = 10       # Zeek/Bro metadata header lines to skip
MAX_ROWS   = 100_000  # Max rows to load per capture file
OUTPUT_CSV = "iot23_combined.csv"

# Capture scenario IDs included in this study
CAPTURE_IDS = [1, 3, 7, 8, 9, 17, 20, 21, 33, 34, 35, 36, 39, 42, 43, 44, 48, 49, 52, 60]

COLUMNS = [
    'ts', 'uid', 'id.orig_h', 'id.orig_p', 'id.resp_h', 'id.resp_p',
    'proto', 'service', 'duration', 'orig_bytes', 'resp_bytes',
    'conn_state', 'local_orig', 'local_resp', 'missed_bytes', 'history',
    'orig_pkts', 'orig_ip_bytes', 'resp_pkts', 'resp_ip_bytes', 'label'
]

# Columns to drop (identifiers, IPs, ports — not useful as ML features)
DROP_COLS = ['ts', 'uid', 'id.orig_h', 'id.orig_p', 'id.resp_h', 'id.resp_p',
             'service', 'local_orig', 'local_resp', 'history']

print(f'Base path : {BASE_PATH}')
print(f'Captures  : {CAPTURE_IDS}')
print(f'Rows/file : {MAX_ROWS:,}')

## 3. Load All 23 Capture Files

In [ ]:
def load_capture(capture_id: int) -> pd.DataFrame:
    """Load a single IoT-23 capture file into a DataFrame."""
    path = os.path.join(
        BASE_PATH,
        f"CTU-IoT-Malware-Capture-{capture_id}-1",
        "bro", "conn.log.labeled"
    )
    df = pd.read_table(
        filepath_or_buffer=path,
        skiprows=SKIP_ROWS,
        nrows=MAX_ROWS
    )
    df.columns = COLUMNS
    df.drop(df.tail(1).index, inplace=True)  # Remove trailing summary row
    df['capture_id'] = capture_id            # Tag source for traceability
    return df

frames = []
for cid in CAPTURE_IDS:
    try:
        df_tmp = load_capture(cid)
        frames.append(df_tmp)
        print(f'  Capture {cid:>2} — {len(df_tmp):>7,} rows loaded')
    except FileNotFoundError:
        print(f'  Capture {cid:>2} — FILE NOT FOUND (skipped)')

df_raw = pd.concat(frames, ignore_index=True)
print(f'\nTotal rows after concat: {len(df_raw):,}')
print(f'Total columns          : {df_raw.shape[1]}')

## 4. Label Cleaning
Raw labels contain prefix strings like `'-   Malicious   '` or `'(empty)   Malicious   '`. Extract the attack type.

In [ ]:
print('Raw label distribution:')
print(df_raw['label'].value_counts().to_string())

In [ ]:
LABEL_MAP = {
    '-   Malicious   PartOfAHorizontalPortScan'   : 'PartOfAHorizontalPortScan',
    '(empty)   Malicious   PartOfAHorizontalPortScan': 'PartOfAHorizontalPortScan',
    '-   Malicious   Okiru'                        : 'Okiru',
    '(empty)   Malicious   Okiru'                  : 'Okiru',
    '-   Benign   -'                               : 'Benign',
    '(empty)   Benign   -'                         : 'Benign',
    '-   Malicious   DDoS'                         : 'DDoS',
    '-   Malicious   C&C'                          : 'C&C',
    '(empty)   Malicious   C&C'                    : 'C&C',
    '-   Malicious   Attack'                       : 'Attack',
    '(empty)   Malicious   Attack'                 : 'Attack',
    '-   Malicious   C&C-HeartBeat'                : 'C&C-HeartBeat',
    '(empty)   Malicious   C&C-HeartBeat'          : 'C&C-HeartBeat',
    '-   Malicious   C&C-FileDownload'             : 'C&C-FileDownload',
    '-   Malicious   C&C-Torii'                    : 'C&C-Torii',
    '-   Malicious   C&C-HeartBeat-FileDownload'   : 'C&C-HeartBeat-FileDownload',
    '-   Malicious   FileDownload'                 : 'FileDownload',
    '-   Malicious   C&C-Mirai'                    : 'C&C-Mirai',
    '-   Malicious   Okiru-Attack'                 : 'Okiru-Attack',
}

df_raw['label'] = df_raw['label'].map(LABEL_MAP).fillna(df_raw['label'])

print('Cleaned label distribution:')
print(df_raw['label'].value_counts().to_string())

## 5. Feature Engineering

In [ ]:
# Drop non-feature columns
df = df_raw.drop(columns=DROP_COLS + ['capture_id'])

# One-hot encode categorical features
df = pd.get_dummies(df, columns=['proto'])
df = pd.get_dummies(df, columns=['conn_state'])

# Replace '-' missing value placeholders with 0
for col in ['duration', 'orig_bytes', 'resp_bytes']:
    df[col] = df[col].astype(str).str.replace('-', '0').astype(float)

# Fill any remaining NaN values
df.fillna(-1, inplace=True)

print('Final shape :', df.shape)
print('Missing values:', df.isna().sum().sum())
print('\nColumns:', df.columns.tolist())

In [ ]:
df.head(5)

## 6. Export Combined Dataset

In [ ]:
df.to_csv(OUTPUT_CSV, index=False)
print(f'Saved → {OUTPUT_CSV}  ({len(df):,} rows × {df.shape[1]} cols)')